# 🚁 Helipoint Detector — Experiment 3

---
### YOLOv8n · Fork dataset (augmented) · São Paulo helipad detection

**PUC-SP · Machine Learning and Computer Vision**

This notebook trains, evaluates, exports and geospatially visualizes a helipad detector.

| | |
|---|---|
| Dataset | `fabiana-campanari-workspace/my-first-project-kfo4l-jcbdq` (fork of the original PUC-SP dataset) |
| Augmentation | rotate ±15°, brightness ±25%, horizontal + vertical flip |
| Model | YOLOv8n, 100 epochs, imgsz=640, seed=42 |
| Outputs | trained weights (`.pt` + `.onnx`), metrics, geospatial map |

> **Attribution:** The base dataset (images, original annotations, SP neighborhoods) was created and
> curated by Pedro Vyctor Almeida. The fork used in this notebook — with additional augmentation
> (rotate ±15°, brightness ±25%, flip H/V) — and all the training/analysis of this exp3 were done
> by Fabiana Campanari, based on his original work.

> **Naming note:** this is **exp3**, a supplementary run — not part of the briefing's official
> exp1-vs-exp2 comparison (which requires both runs on the *same* dataset). exp1 and exp2 are
> tracked separately in `yolo_training_exp1.ipynb` / `yolo_training_exp2.ipynb` and in the
> main README. This notebook is kept useful in its own right — it's the model actually powering
> the Roboflow workflow and the Streamlit dashboard.

**Contents:** 1) Setup 2) Train 3) Export ONNX 4) Compare vs. exp1 5) Sample inference 6) Geospatial map (Kepler.gl)


## 1️⃣ Setup

---


In [ ]:
import torch

def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = select_device()
print(f"Using device: {device}")


**Roboflow API key** — read from Colab Secrets (🔒 icon in the sidebar) with a `getpass` fallback, never hardcoded.


In [ ]:
import os

def get_roboflow_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ROBOFLOW_API_KEY')
    if key:
        return key
    from getpass import getpass
    return getpass('Enter your Roboflow API key: ')

os.environ['ROBOFLOW_API_KEY'] = get_roboflow_api_key()


In [ ]:
import sys
!{sys.executable} -m pip install -q roboflow ultralytics keplergl


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

EXPECTED_WORKSPACE = "fabiana-campanari-workspace"
EXPECTED_PROJECT = "my-first-project-kfo4l-jcbdq"

project = rf.workspace(EXPECTED_WORKSPACE).project(EXPECTED_PROJECT)
assert project.id.endswith(f"{EXPECTED_WORKSPACE}/{EXPECTED_PROJECT}"), (
    f"Wrong dataset! Expected {EXPECTED_WORKSPACE}/{EXPECTED_PROJECT}, got {project.id}. "
    "This is exp3 — it must use the FORK, not the original dataset (that's exp1/exp2)."
)

version = project.version(1)
dataset = version.download("yolov8")
print(f"Downloaded to: {dataset.location}")


## 2️⃣ Train

---


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=device,
    seed=42,
    project='runs/detect',
    name='exp3',
)

# Ultralytics is the source of truth for the save path — never hardcode/guess it
# (a guessed path caused a real bug earlier in this project's history).
save_dir = str(results.save_dir)
best_weights = f'{save_dir}/weights/best.pt'
print('Results saved to:', save_dir)
print('Best weights:', best_weights)


## 3️⃣ Export to ONNX

---

Parameters below were validated in a real prior run of this pipeline.


In [ ]:
onnx_path = model.export(format='onnx', opset=12, simplify=True, imgsz=640)
print('ONNX model exported to:', onnx_path)


## 4️⃣ Compare against exp1

---

> ⚠️ **Data integrity note:** an earlier version of this notebook fabricated a placeholder `exp1_results.csv` when the real file wasn't present in the Colab session, and printed a comparison against that synthetic data. This version never does that — it either reads the **real** `exp1/results.csv` if you've placed it alongside this notebook (e.g. by cloning the repo), or falls back to the real, verified exp1 numbers hardcoded below (computed directly from the actual `results.csv` — see the project README). It never invents numbers.


In [ ]:
import pandas as pd
from pathlib import Path

# Real exp1 numbers (YOLOv8n, 60 epochs, original PUC-SP dataset) -- computed from the
# actual artifacts/runs/runs/detect/exp1/results.csv in the repo. Used as a verified
# fallback only; if that file is locally available, it is read directly instead.
EXP1_REFERENCE = {
    'best_epoch': 54, 'best_precision': 1.000, 'best_recall': 0.9626,
    'best_map50': 0.9939, 'best_map50_95': 0.8812,
    'final_epoch': 60, 'final_precision': 0.9921, 'final_recall': 0.9706,
    'final_map50': 0.9939, 'final_map50_95': 0.8409,
}

# Real exp3 numbers (YOLOv8n, 100 epochs, FORK dataset) -- this run already
# happened and was confirmed against artifacts/runs/detect/exp3/results.csv
# (renamed from a folder that was mistakenly called "exp2" on disk). Used as
# a verified fallback so this notebook does not need to retrain from scratch
# just to show the comparison.
EXP3_REFERENCE = {
    'best_epoch': 78, 'best_precision': 0.9426, 'best_recall': 1.0000,
    'best_map50': 0.9930, 'best_map50_95': 0.8853,
    'final_epoch': 100, 'final_precision': 0.9682, 'final_recall': 0.9706,
    'final_map50': 0.9515, 'final_map50_95': 0.8415,
}

exp1_csv_candidates = [
    Path('artifacts/runs/runs/detect/exp1/results.csv'),
    Path('artifacts/runs/detect/exp1/exp1_results.csv'),
    Path('../exp1/results.csv'),
    Path('exp1_results.csv'),
]
exp1_path = next((p for p in exp1_csv_candidates if p.exists()), None)

if exp1_path is not None:
    df1 = pd.read_csv(exp1_path)
    df1.columns = df1.columns.str.strip()
    b1 = df1.loc[df1['metrics/mAP50-95(B)'].idxmax()]
    f1 = df1.iloc[-1]
    exp1 = {
        'best_epoch': int(b1['epoch']), 'best_precision': b1['metrics/precision(B)'],
        'best_recall': b1['metrics/recall(B)'], 'best_map50': b1['metrics/mAP50(B)'],
        'best_map50_95': b1['metrics/mAP50-95(B)'], 'final_epoch': int(f1['epoch']),
        'final_precision': f1['metrics/precision(B)'], 'final_recall': f1['metrics/recall(B)'],
        'final_map50': f1['metrics/mAP50(B)'], 'final_map50_95': f1['metrics/mAP50-95(B)'],
    }
    print(f'Loaded REAL exp1 data from {exp1_path}')
else:
    exp1 = EXP1_REFERENCE
    print('exp1 results.csv not found locally -- using the verified reference numbers above '
          '(NOT synthetic data -- these are the real published exp1 metrics).')

exp3_csv_candidates = [
    Path(f'{save_dir}/results.csv') if 'save_dir' in dir() else None,
    Path('artifacts/runs/detect/exp3/results.csv'),
    Path('artifacts/runs/runs/detect/exp3/results.csv'),
]
exp3_path = next((p for p in exp3_csv_candidates if p is not None and p.exists()), None)

if exp3_path is not None:
    df3 = pd.read_csv(exp3_path)
    df3.columns = df3.columns.str.strip()
    b3 = df3.loc[df3['metrics/mAP50-95(B)'].idxmax()]
    f3 = df3.iloc[-1]
    exp3 = {
        'best_precision': b3['metrics/precision(B)'], 'best_recall': b3['metrics/recall(B)'],
        'best_map50': b3['metrics/mAP50(B)'], 'best_map50_95': b3['metrics/mAP50-95(B)'],
        'final_precision': f3['metrics/precision(B)'], 'final_recall': f3['metrics/recall(B)'],
        'final_map50': f3['metrics/mAP50(B)'], 'final_map50_95': f3['metrics/mAP50-95(B)'],
    }
    print(f'Loaded REAL exp3 data from {exp3_path}')
else:
    exp3 = EXP3_REFERENCE
    print('exp3 results.csv not found locally -- using the verified reference numbers above '
          '(this run already happened once; these are its real, confirmed metrics).')

comparison = pd.DataFrame({
    'exp1 (original ds, 60ep) - best': [exp1['best_precision'], exp1['best_recall'], exp1['best_map50'], exp1['best_map50_95']],
    'exp1 - final': [exp1['final_precision'], exp1['final_recall'], exp1['final_map50'], exp1['final_map50_95']],
    'exp3 (fork ds, 100ep) - best': [exp3['best_precision'], exp3['best_recall'], exp3['best_map50'], exp3['best_map50_95']],
    'exp3 - final': [exp3['final_precision'], exp3['final_recall'], exp3['final_map50'], exp3['final_map50_95']],
}, index=['Precision', 'Recall', 'mAP@50', 'mAP@50-95'])

print('\nNote: exp1 and exp3 trained on DIFFERENT dataset versions/augmentation -- this is a '
      'reference point, not an apples-to-apples ablation (that comparison is exp1 vs exp2).\n')
display(comparison.style.format('{:.4f}'))


## 5️⃣ Sample inference

---


In [ ]:
import os
from pathlib import Path
from IPython.display import Image, display

val_images_path = Path(dataset.location) / 'valid' / 'images'
sample_images = sorted(val_images_path.glob('*.jpg')) if val_images_path.exists() else []

if sample_images:
    sample = sample_images[0]
    print(f'Running inference on: {sample.name}')
    pred = model.predict(source=str(sample), save=True, conf=0.5)
    predict_dirs = sorted(Path('runs/detect').glob('predict*'))
    if predict_dirs:
        result_path = predict_dirs[-1] / sample.name
        if result_path.exists():
            display(Image(filename=str(result_path)))
else:
    print(f'No validation images found at {val_images_path}')


## 6️⃣ Geospatial visualization — Kepler.gl

---

Point layer + density heatmap over the helipad discovery dataset (`src/geospatial/helipad_coordinates_bbox.csv`). Renders reliably by embedding the **saved HTML file** explicitly via `IPython.display.HTML`, rather than relying on `KeplerGl._repr_html_()` (unreliable outside a live Jupyter kernel — the root cause of the blank-map issue in earlier versions of this notebook).


In [ ]:
import re
import json
import numpy as np
import pandas as pd
from keplergl import KeplerGl
from IPython.display import HTML

csv_input_path  = 'src/geospatial/helipad_coordinates_bbox.csv'
csv_output_path = 'src/geospatial/helipad_coordinates_processed.csv'
config_output   = 'src/geospatial/keplergl_map_config.json'
html_output      = 'src/geospatial/keplergl_map_loaded.html'

def dms_to_decimal(d, m, s, direction):
    dd = float(d) + float(m) / 60 + float(s) / 3600
    return -dd if direction in ('S', 'W') else dd

def parse_dms_pair(text):
    m = re.match(r"(\d+)°(\d+)'([\d.]+)\"([NS])\s+(\d+)°(\d+)'([\d.]+)\"([EW])", text)
    if not m:
        return (np.nan,) * 4
    lat_d, lat_m, lat_s, lat_dir, lon_d, lon_m, lon_s, lon_dir = m.groups()
    lat = dms_to_decimal(lat_d, lat_m, lat_s, lat_dir)
    lon = dms_to_decimal(lon_d, lon_m, lon_s, lon_dir)
    return lon, lat, lon, lat

def parse_numeric_bbox(text):
    parts = text.replace(',', ' ').split()
    if len(parts) != 4:
        return (np.nan,) * 4
    try:
        return tuple(float(p) for p in parts)
    except ValueError:
        return (np.nan,) * 4

def parse_bbox(text):
    text = str(text)
    return parse_dms_pair(text) if '°' in text else parse_numeric_bbox(text)

df = pd.read_csv(csv_input_path)
parsed = df['Coordenadas da Bounding Box'].apply(parse_bbox)
df[['min_longitude', 'min_latitude', 'max_longitude', 'max_latitude']] = pd.DataFrame(
    parsed.tolist(), index=df.index
)
df['center_latitude'] = (df['min_latitude'] + df['max_latitude']) / 2
df['center_longitude'] = (df['min_longitude'] + df['max_longitude']) / 2
df.to_csv(csv_output_path, index=False)

print(f'Total helipads: {len(df)}  |  parse failures: {df["center_latitude"].isna().sum()}')
print(f'Map center: lat={df["center_latitude"].mean():.4f}, lon={df["center_longitude"].mean():.4f}')

config = {
    'mapState': {
        'latitude': float(df['center_latitude'].mean()),
        'longitude': float(df['center_longitude'].mean()),
        'zoom': 10,
    },
    'visState': {
        'layers': [
            {'id': 'helipad_points', 'type': 'point', 'config': {
                'dataId': 'Helipad Coords', 'label': 'Heliportos — Centroides',
                'color': [0, 206, 209],
                'columns': {'lat': 'center_latitude', 'lng': 'center_longitude'},
                'isVisible': True,
                'visConfig': {
                    'radius': 10, 'opacity': 0.8, 'outline': False,
                    'colorRange': {'name': 'Global Warming', 'type': 'sequential',
                        'colors': ['#5A1846', '#900C3F', '#C70039', '#E3611C', '#F1920E', '#FFC300']},
                    'radiusRange': [0, 50],
                },
            }},
            {'id': 'helipad_heatmap', 'type': 'heatmap', 'config': {
                'dataId': 'Helipad Coords', 'label': 'Densidade de Heliportos',
                'color': [255, 0, 0],
                'columns': {'lat': 'center_latitude', 'lng': 'center_longitude'},
                'isVisible': True,
                'visConfig': {'opacity': 0.8, 'radius': 10,
                    'colorRange': {'name': 'Custom Hot', 'type': 'sequential',
                        'colors': ['#FFF8DC', '#FFE8C1', '#FFD8A6', '#FFC88B', '#FFB870',
                                   '#FFA855', '#FF983A', '#FF881F', '#FF7804', '#FF6800']}},
            }},
        ],
        'interactionConfig': {'tooltip': {'fieldsToShow': {
            'Helipad Coords': ['Nome do Bairro', 'Coordenadas da Bounding Box']}}},
    },
    'mapStyle': {
        'styleType': 'dark', 'topLayerGroups': {},
        'visibleLayerGroups': {'label': True, 'road': True, 'border': False,
                                'building': True, 'water': True, 'land': True},
        'buildingColor': [19, 19, 19], 'backgroundColor': [19, 19, 19],
    },
}

with open(config_output, 'w') as f:
    json.dump(config, f, indent=2)
print('Config saved:', config_output)

kmap = KeplerGl(height=600, config=config)
kmap.add_data(data=df, name='Helipad Coords')
kmap.save_to_html(file_name=html_output)
print('HTML saved:', html_output)

with open(html_output, 'r', encoding='utf-8') as f:
    display(HTML(f.read()))


## ✅ Summary — artifacts generated

---


In [ ]:
from IPython.display import Markdown, display

summary = f"""
| Artifact | Path |
|---|---|
| Trained weights | `{save_dir}/weights/best.pt` |
| ONNX export | `{onnx_path}` |
| Metrics | `{save_dir}/results.csv` |
| Kepler config | `src/geospatial/keplergl_map_config.json` |
| Kepler map (standalone) | `src/geospatial/keplergl_map_loaded.html` |

**Next steps:** zip and download `{save_dir}` for local artifacts; upload `best.pt`/`.onnx` to 
Roboflow to power the "Helipad Detector." workflow; commit the Kepler files to 
`src/geospatial/` for the dashboard's Map tab.
"""
display(Markdown(summary))


In [ ]:
import shutil

shutil.make_archive('exp3_zipped', 'zip', save_dir)
print('Created exp3_zipped.zip')

try:
    from google.colab import files
    files.download('exp3_zipped.zip')
except ImportError:
    print('Not running in Colab — find exp3_zipped.zip in the current working directory.')
